In [2]:
# ============================================================================
# CELL 4: Create Conversion Script (Save as convert_dgl_to_pt.py)
# ============================================================================

"""
DGL → PyTorch Tensor Conversion Script
--------------------------------------
Environment: dgl_only (NO PyG needed!)
Input: DGL .bin files
Output: PyTorch .pt files (readable by PyG)
"""

import os
import torch
import dgl
from tqdm import tqdm

def dgl_to_tensors(dgl_graph):
    """
    Extract raw tensors from DGL graph
    Returns a dictionary of PyTorch tensors
    """
    # Extract node features
    node_features = dgl_graph.ndata['feats'].float()
    
    # Extract edges (convert to COO format for PyG)
    src, dst = dgl_graph.edges()
    edge_index = torch.stack([src, dst], dim=0).long()
    
    # Extract edge features
    edge_features = dgl_graph.edata['feats'].float()
    
    # Extract Laplacian positional encoding
    laplacian_pe = dgl_graph.ndata['lap_pos_enc'].float()
    
    # Return as dictionary (not PyG Data object - we don't have PyG here!)
    return {
        'x': node_features,
        'edge_index': edge_index,
        'edge_attr': edge_features,
        'laplacian_pe': laplacian_pe
    }


def convert_all_proteins():
    """
    Convert all DGL protein graphs to PyTorch tensor format
    """
    # Paths
    dgl_dir = 'data/Davis/processed/pocket_graph'
    output_dir = 'proteins'  # Output directory
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all protein .bin files
    protein_files = sorted([f for f in os.listdir(dgl_dir) if f.endswith('.bin')])
    
    print("="*60)
    print("DGL → PyTorch Tensor Conversion")
    print("="*60)
    print(f"Input:  {dgl_dir}/")
    print(f"Output: {output_dir}/")
    print(f"Total:  {len(protein_files)} proteins")
    print("="*60)
    
    success_count = 0
    failed_proteins = []
    
    for protein_file in tqdm(protein_files, desc="Converting"):
        protein_id = protein_file.replace('.bin', '')
        
        try:
            # Load DGL graph
            dgl_path = os.path.join(dgl_dir, protein_file)
            dgl_graphs, _ = dgl.load_graphs(dgl_path)
            dgl_graph = dgl_graphs[0]
            
            # Convert to tensors (dictionary)
            tensor_dict = dgl_to_tensors(dgl_graph)
            
            # Save as .pt file
            save_path = os.path.join(output_dir, f'{protein_id}.pt')
            torch.save(tensor_dict, save_path)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n⚠️ Error converting {protein_id}: {e}")
            failed_proteins.append(protein_id)
            continue
    
    print("\n" + "="*60)
    print("✅ Conversion Complete!")
    print("="*60)
    print(f"Success: {success_count}/{len(protein_files)}")
    print(f"Failed:  {len(failed_proteins)}")
    if failed_proteins:
        print(f"Failed IDs: {failed_proteins}")
    print("="*60)
    
    return success_count, failed_proteins


def verify_conversion():
    """
    Verify conversion by comparing DGL and tensor versions
    """
    print("\n" + "="*60)
    print("🔍 Verification")
    print("="*60)
    
    # Pick first available protein
    dgl_dir = 'data/Davis/processed/pocket_graph'
    protein_files = [f for f in os.listdir(dgl_dir) if f.endswith('.bin')]
    
    if not protein_files:
        print("❌ No protein files found!")
        return
    
    test_protein = protein_files[0].replace('.bin', '')
    
    # Load DGL graph
    dgl_path = f'data/Davis/processed/pocket_graph/{test_protein}.bin'
    dgl_graphs, _ = dgl.load_graphs(dgl_path)
    dgl_graph = dgl_graphs[0]
    
    # Load tensor dict
    tensor_dict = torch.load(f'proteins/{test_protein}.pt')
    
    print(f"\nTest Protein: {test_protein}")
    print(f"\nDGL Graph:")
    print(f"  Nodes: {dgl_graph.num_nodes()}")
    print(f"  Edges: {dgl_graph.num_edges()}")
    print(f"  Node features: {dgl_graph.ndata['feats'].shape}")
    
    print(f"\nTensor Dict:")
    print(f"  Nodes: {tensor_dict['x'].shape[0]}")
    print(f"  Edges: {tensor_dict['edge_index'].shape[1]}")
    print(f"  Node features: {tensor_dict['x'].shape}")
    print(f"  Edge features: {tensor_dict['edge_attr'].shape}")
    print(f"  Laplacian PE: {tensor_dict['laplacian_pe'].shape}")
    
    # Verify they match
    nodes_match = dgl_graph.num_nodes() == tensor_dict['x'].shape[0]
    edges_match = dgl_graph.num_edges() == tensor_dict['edge_index'].shape[1]
    features_match = torch.allclose(dgl_graph.ndata['feats'].float(), tensor_dict['x'])
    
    print(f"\n✓ Validation:")
    print(f"  Nodes match: {nodes_match}")
    print(f"  Edges match: {edges_match}")
    print(f"  Features match: {features_match}")
    
    if all([nodes_match, edges_match, features_match]):
        print("\n✅ Conversion verified successfully!")
    else:
        print("\n⚠️ Conversion may have issues!")
    
    print("="*60)


if __name__ == "__main__":
    # Convert all proteins
    success, failed = convert_all_proteins()
    
    # Verify conversion
    if success > 0:
        verify_conversion()
    
    print("\n" + "="*60)
    print("🎉 Done! Switch to cse_330_cpu to use these files")
    print("="*60)
    print("\nNext:")
    print("  1. conda activate cse_330_cpu")
    print("  2. Load with: torch.load('proteins/PROTEIN_ID.pt')")
    print("  3. Wrap in PyG Data object if needed")
    print("="*60)

DGL → PyTorch Tensor Conversion
Input:  data/Davis/processed/pocket_graph/
Output: proteins/
Total:  379 proteins


Converting: 100%|██████████| 379/379 [00:00<00:00, 792.37it/s]


✅ Conversion Complete!
Success: 379/379
Failed:  0

🔍 Verification

Test Protein: P45984

DGL Graph:
  Nodes: 96
  Edges: 2302
  Node features: torch.Size([96, 41])

Tensor Dict:
  Nodes: 96
  Edges: 2302
  Node features: torch.Size([96, 41])
  Edge features: torch.Size([2302, 5])
  Laplacian PE: torch.Size([96, 8])

✓ Validation:
  Nodes match: True
  Edges match: True
  Features match: True

✅ Conversion verified successfully!

🎉 Done! Switch to cse_330_cpu to use these files

Next:
  1. conda activate cse_330_cpu
  2. Load with: torch.load('proteins/PROTEIN_ID.pt')
  3. Wrap in PyG Data object if needed


In [2]:
# ============================================================================
# CELL 7: Test Loading in PyG Environment (create test_loading.py)
# ============================================================================

"""
Test loading converted proteins in PyG environment
NO DGL NEEDED HERE!
"""

import torch
import numpy as np
import os
from torch_geometric.data import Data

def test_load_proteins():
    """
    Test loading converted proteins (no DGL needed!)
    """
    print("="*60)
    print("Testing Protein Loading (PyG Environment)")
    print("="*60)
    
    # Get list of proteins
    proteins_dir = 'proteins'
    protein_files = sorted([f for f in os.listdir(proteins_dir) if f.endswith('.pt')])
    
    print(f"\nAvailable proteins: {len(protein_files)}")
    
    # Test loading first 3 proteins
    for i in range(1, len(protein_files)):
        protein_id = protein_files[i].replace('.pt', '')
        print(f"\n{'='*40}")
        print(f"Loading: {protein_id}")
        
        # Load tensor dictionary
        tensor_dict = torch.load(f'proteins/{protein_id}.pt')
        
        # Wrap in PyG Data object
        protein_graph = Data(
            x=tensor_dict['x'],
            edge_index=tensor_dict['edge_index'],
            edge_attr=tensor_dict['edge_attr'],
            laplacian_pe=tensor_dict['laplacian_pe']
        )
        
        print(f"  Nodes: {protein_graph.x.shape[0]}")
        print(f"  Node features: {protein_graph.x.shape[1]}")
        print(f"  Edges: {protein_graph.edge_index.shape[1]}")
        print(f"  Edge features: {protein_graph.edge_attr.shape[1]}")
        print(f"  Laplacian PE: {protein_graph.laplacian_pe.shape[1]}")
        
        # Also load ESM embedding
        esm_path = f'data/Davis/processed/ESM_embedding_pocket/{protein_id}.npy'
        if os.path.exists(esm_path):
            esm_embedding = np.load(esm_path, allow_pickle=True)
            print(f"  ESM embedding: {esm_embedding.shape}")
    
    print("\n" + "="*60)
    print("✅ SUCCESS! All proteins load perfectly!")
    print("="*60)

if __name__ == "__main__":
    test_load_proteins()

Testing Protein Loading (PyG Environment)

Available proteins: 379

Loading: A0A2K5TT62
  Nodes: 27
  Node features: 41
  Edges: 116
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (27, 320)

Loading: A0A6D2XP67
  Nodes: 9
  Node features: 41
  Edges: 30
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (9, 320)

Loading: A0A6D2Y6E9
  Nodes: 22
  Node features: 41
  Edges: 110
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (22, 320)

Loading: A1R9W2
  Nodes: 10
  Node features: 41
  Edges: 68
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (10, 320)

Loading: A4VCI5
  Nodes: 23
  Node features: 41
  Edges: 154
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (23, 320)

Loading: B3KS12
  Nodes: 12
  Node features: 41
  Edges: 104
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (12, 320)

Loading: E9PUP9
  Nodes: 129
  Node features: 41
  Edges: 3150
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (129, 320)

Loading: L7RXH5
  Nodes: 33
  Node feat

In [1]:
# ============================================================================
# convert_dgl_to_pt.py (CORRECTED VERSION)
# ============================================================================

"""
DGL → PyTorch Tensor Conversion Script
--------------------------------------
Environment: dgl_only (NO PyG needed!)
Input: DGL .bin files
Output: PyTorch .pt files (readable by PyG)
"""

import os
import torch
import dgl
from tqdm import tqdm

def dgl_to_tensors(dgl_graph):
    """
    Extract raw tensors from DGL graph
    Returns a dictionary of PyTorch tensors
    """
    # Extract node features
    node_features = dgl_graph.ndata['feats'].float()
    
    # Extract edges (convert to COO format for PyG)
    src, dst = dgl_graph.edges()
    edge_index = torch.stack([src, dst], dim=0).long()
    
    # Extract edge features
    edge_features = dgl_graph.edata['feats'].float()
    
    # Extract Laplacian positional encoding
    laplacian_pe = dgl_graph.ndata['lap_pos_enc'].float()
    
    # IMPORTANT: Use 'lap_pos_enc' (not 'laplacian_pe') to match your model!
    return {
        'x': node_features,
        'edge_index': edge_index,
        'edge_attr': edge_features,
        'lap_pos_enc': laplacian_pe  # ← Changed from 'laplacian_pe'
    }


def convert_all_proteins():
    """
    Convert all DGL protein graphs to PyTorch tensor format
    """
    # Paths
    dgl_dir = 'data/Davis/processed/pocket_graph'
    output_dir = 'proteins'  # Output directory
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all protein .bin files
    protein_files = sorted([f for f in os.listdir(dgl_dir) if f.endswith('.bin')])
    
    print("="*60)
    print("DGL → PyTorch Tensor Conversion")
    print("="*60)
    print(f"Input:  {dgl_dir}/")
    print(f"Output: {output_dir}/")
    print(f"Total:  {len(protein_files)} proteins")
    print("="*60)
    
    success_count = 0
    failed_proteins = []
    
    for protein_file in tqdm(protein_files, desc="Converting"):
        protein_id = protein_file.replace('.bin', '')
        
        try:
            # Load DGL graph
            dgl_path = os.path.join(dgl_dir, protein_file)
            dgl_graphs, _ = dgl.load_graphs(dgl_path)
            dgl_graph = dgl_graphs[0]
            
            # Convert to tensors (dictionary)
            tensor_dict = dgl_to_tensors(dgl_graph)
            
            # Save as .pt file
            save_path = os.path.join(output_dir, f'{protein_id}.pt')
            torch.save(tensor_dict, save_path)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n⚠️ Error converting {protein_id}: {e}")
            failed_proteins.append(protein_id)
            continue
    
    print("\n" + "="*60)
    print("✅ Conversion Complete!")
    print("="*60)
    print(f"Success: {success_count}/{len(protein_files)}")
    print(f"Failed:  {len(failed_proteins)}")
    if failed_proteins:
        print(f"Failed IDs: {failed_proteins}")
    print("="*60)
    
    return success_count, failed_proteins


def verify_conversion():
    """
    Verify conversion by comparing DGL and tensor versions
    """
    print("\n" + "="*60)
    print("🔍 Verification")
    print("="*60)
    
    # Pick first available protein
    dgl_dir = 'data/Davis/processed/pocket_graph'
    protein_files = [f for f in os.listdir(dgl_dir) if f.endswith('.bin')]
    
    if not protein_files:
        print("❌ No protein files found!")
        return
    
    test_protein = protein_files[0].replace('.bin', '')
    
    # Load DGL graph
    dgl_path = f'data/Davis/processed/pocket_graph/{test_protein}.bin'
    dgl_graphs, _ = dgl.load_graphs(dgl_path)
    dgl_graph = dgl_graphs[0]
    
    # Load tensor dict
    tensor_dict = torch.load(f'proteins/{test_protein}.pt')
    
    print(f"\nTest Protein: {test_protein}")
    print(f"\nDGL Graph:")
    print(f"  Nodes: {dgl_graph.num_nodes()}")
    print(f"  Edges: {dgl_graph.num_edges()}")
    print(f"  Node features: {dgl_graph.ndata['feats'].shape}")
    
    print(f"\nTensor Dict Keys: {list(tensor_dict.keys())}")
    print(f"  Nodes: {tensor_dict['x'].shape[0]}")
    print(f"  Edges: {tensor_dict['edge_index'].shape[1]}")
    print(f"  Node features: {tensor_dict['x'].shape}")
    print(f"  Edge features: {tensor_dict['edge_attr'].shape}")
    print(f"  Laplacian PE: {tensor_dict['lap_pos_enc'].shape}")  # ← Changed
    
    # Verify they match
    nodes_match = dgl_graph.num_nodes() == tensor_dict['x'].shape[0]
    edges_match = dgl_graph.num_edges() == tensor_dict['edge_index'].shape[1]
    features_match = torch.allclose(dgl_graph.ndata['feats'].float(), tensor_dict['x'])
    
    print(f"\n✓ Validation:")
    print(f"  Nodes match: {nodes_match}")
    print(f"  Edges match: {edges_match}")
    print(f"  Features match: {features_match}")
    
    if all([nodes_match, edges_match, features_match]):
        print("\n✅ Conversion verified successfully!")
    else:
        print("\n⚠️ Conversion may have issues!")
    
    print("="*60)


if __name__ == "__main__":
    # Convert all proteins
    success, failed = convert_all_proteins()
    
    # Verify conversion
    if success > 0:
        verify_conversion()
    
    print("\n" + "="*60)
    print("🎉 Done! Switch to cse_330_cpu to use these files")
    print("="*60)
    print("\nNext:")
    print("  1. conda activate cse_330_cpu")
    print("  2. Load with: torch.load('proteins/PROTEIN_ID.pt')")
    print("  3. Wrap in PyG Data object")
    print("="*60)

DGL → PyTorch Tensor Conversion
Input:  data/Davis/processed/pocket_graph/
Output: proteins/
Total:  379 proteins


Converting: 100%|██████████| 379/379 [00:00<00:00, 1174.95it/s]


✅ Conversion Complete!
Success: 379/379
Failed:  0

🔍 Verification

Test Protein: P45984

DGL Graph:
  Nodes: 96
  Edges: 2302
  Node features: torch.Size([96, 41])

Tensor Dict Keys: ['x', 'edge_index', 'edge_attr', 'lap_pos_enc']
  Nodes: 96
  Edges: 2302
  Node features: torch.Size([96, 41])
  Edge features: torch.Size([2302, 5])
  Laplacian PE: torch.Size([96, 8])

✓ Validation:
  Nodes match: True
  Edges match: True
  Features match: True

✅ Conversion verified successfully!

🎉 Done! Switch to cse_330_cpu to use these files

Next:
  1. conda activate cse_330_cpu
  2. Load with: torch.load('proteins/PROTEIN_ID.pt')
  3. Wrap in PyG Data object


In [1]:
# ============================================================================
# test_loading.py (CORRECTED VERSION)
# ============================================================================

"""
Test loading converted proteins in PyG environment
NO DGL NEEDED HERE!
"""

import torch
import numpy as np
import os
from torch_geometric.data import Data

def test_load_proteins():
    """
    Test loading converted proteins (no DGL needed!)
    """
    print("="*60)
    print("Testing Protein Loading (PyG Environment)")
    print("="*60)
    
    # Get list of proteins
    proteins_dir = 'proteins'
    protein_files = sorted([f for f in os.listdir(proteins_dir) if f.endswith('.pt')])
    
    print(f"\nAvailable proteins: {len(protein_files)}")
    
    # Test loading first 3 proteins
    for i in range(min(3, len(protein_files))):
        protein_id = protein_files[i].replace('.pt', '')
        print(f"\n{'='*40}")
        print(f"Loading: {protein_id}")
        
        # Load tensor dictionary
        tensor_dict = torch.load(f'proteins/{protein_id}.pt')
        
        print(f"  Dict keys: {list(tensor_dict.keys())}")
        
        # Wrap in PyG Data object (using correct attribute name)
        protein_graph = Data(
            x=tensor_dict['x'],
            edge_index=tensor_dict['edge_index'],
            edge_attr=tensor_dict['edge_attr'],
            lap_pos_enc=tensor_dict['lap_pos_enc']  # ← Changed from 'laplacian_pe'
        )
        
        print(f"  Nodes: {protein_graph.x.shape[0]}")
        print(f"  Node features: {protein_graph.x.shape[1]}")
        print(f"  Edges: {protein_graph.edge_index.shape[1]}")
        print(f"  Edge features: {protein_graph.edge_attr.shape[1]}")
        print(f"  Laplacian PE: {protein_graph.lap_pos_enc.shape[1]}")
        
        # Verify attribute access works
        assert hasattr(protein_graph, 'lap_pos_enc'), "Missing lap_pos_enc!"
        
        # Also load ESM embedding
        esm_path = f'data/Davis/processed/ESM_embedding_pocket/{protein_id}.npy'
        if os.path.exists(esm_path):
            esm_embedding = np.load(esm_path, allow_pickle=True)
            print(f"  ESM embedding: {esm_embedding.shape}")
    
    print("\n" + "="*60)
    print("✅ SUCCESS! All proteins load perfectly!")
    print("="*60)

if __name__ == "__main__":
    test_load_proteins()

Testing Protein Loading (PyG Environment)

Available proteins: 379

Loading: A0A091FTP9
  Dict keys: ['x', 'edge_index', 'edge_attr', 'lap_pos_enc']
  Nodes: 34
  Node features: 41
  Edges: 350
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (34, 320)

Loading: A0A2K5TT62
  Dict keys: ['x', 'edge_index', 'edge_attr', 'lap_pos_enc']
  Nodes: 27
  Node features: 41
  Edges: 116
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (27, 320)

Loading: A0A6D2XP67
  Dict keys: ['x', 'edge_index', 'edge_attr', 'lap_pos_enc']
  Nodes: 9
  Node features: 41
  Edges: 30
  Edge features: 5
  Laplacian PE: 8
  ESM embedding: (9, 320)

✅ SUCCESS! All proteins load perfectly!
